In [0]:
base_path = "/Volumes/databricks-pyspark/spark_ns/ns_volume/water_mark"
source_path = base_path + "source"
schema_path = base_path + "schema_path"
checkpoint_path = base_path + "checkpoint_path"

In [0]:
dbutils.fs.mkdirs(source_path)

In [0]:
initial_df = spark.readStream \
                  .format("csv") \
                  .option("header","true") \
                      .schema("txn_id INT, event_time TIMESTAMP") \
                          .load(source_path)    

In [0]:
#that means any data that is comming after max time of the event - 10 minutes will accept 
from pyspark.sql.functions import window
max_time = 10 
agg_df = initial_df.withWatermark("event_time",f"{max_time} minutes") \
          .groupBy(window("event_time","5 minutes") ) \
        .count()


In [0]:
dbutils.fs.rm(checkpoint_path, True)

In [0]:
query = agg_df.writeStream \
                 .format("memory") \
                     .outputMode("update") \
                         .queryName("stream_output") \
                             .option("checkpointLocation",checkpoint_path) \
                                 .trigger(availableNow=True)\
                                 .start()

In [0]:
dbutils.fs.put(
    source_path + "/batch_01.csv",
    """txn_id,event_time
1,2025-06-01 10:00:00
2,2025-06-01 10:01:00
3,2025-06-01 10:03:00
4,2025-06-01 10:07:00
""",
    True
)

In [0]:
from pyspark.sql.functions import *

In [0]:
spark.sql("""
          select window.start,
          window.end,
          count from stream_output
          order by window.start
          """).show(truncate=True)

In [0]:
dbutils.fs.put(
    source_path + "/batch_023.csv",
    """txn_id,event_time
5,2025-06-01 10:20:00
6,2025-06-01 10:22:00
""",
    True
)

In [0]:
dbutils.fs.put(
    source_path + "/batch_04_too_late.csv",
    """txn_id,event_time
8,2025-06-01 10:05:00
""",
    True
)